In [1]:
import numpy as np
import sympy as sp

# Símbolos
x, y = sp.symbols('x y')

# Polinomios dados
p_pol_x = x**2 + x + 3   # p^P(x)
p_pol_y = y + 1          # p^G(y)


def coordenadas_en_base(polinomio, variable, base):
    """Vector de coordenadas de 'polinomio' en la base 'base' (lista de polinomios)."""
    grado = len(base) - 1
    M = sp.Matrix([[sp.expand(b).coeff(variable, k) for b in base] for k in range(grado + 1)])
    b = sp.Matrix([sp.expand(polinomio).coeff(variable, k) for k in range(grado + 1)])
    a = M.LUsolve(b)
    return sp.simplify(a)

def producto_exterior(a, b):
    """Producto exterior de vectores de coeficientes a ⊗ b en forma de matriz (i,j)."""
    A = sp.Matrix(a)
    B = sp.Matrix(b)
    return sp.Matrix([[sp.simplify(A[i]*B[j]) for j in range(B.shape[0])] for i in range(A.shape[0])])

def tensor_desde_coeficientes(C, base_x, base_y):
    """Reconstruye el polinomio en dos variables desde C y las bases tensoriales."""
    expr = 0
    for i, bx in enumerate(base_x):
        for j, by in enumerate(base_y):
            expr += C[i, j] * sp.expand(sp.sympify(bx)) * sp.expand(sp.sympify(by))
    return sp.expand(expr)

# ---------------- (a) y (b): base monomial ----------------
base_x_m = [sp.Integer(1), x, x**2]
base_y_m = [sp.Integer(1), y, y**2]

a_m = coordenadas_en_base(p_pol_x, x, base_x_m)   # [3, 1, 1]^T
b_m = coordenadas_en_base(p_pol_y, y, base_y_m)   # [1, 1, 0]^T

C_m = producto_exterior(a_m, b_m)              # Matriz c^{ij}
p_tensor = tensor_desde_coeficientes(C_m, base_x_m, base_y_m)

print("=== (a) y (b) Base monomial {1,x,x^2} ⊗ {1,y,y^2} ===")
print("Coeficientes de p^P(x):", list(a_m))
print("Coeficientes de p^G(y):", list(b_m))
print("Matriz c^{ij} (i para x^i; j para y^j):")
sp.pprint(C_m)
print("\nPolinomio tensor p^{P⊗G}(x,y) expandido:")
sp.pprint(p_tensor)

# ---------------- (c): base de Legendre en x ----------------
P0x, P1x, P2x = sp.legendre(0, x), sp.legendre(1, x), sp.legendre(2, x)
base_x_L = [P0x, P1x, P2x]

a_L = coordenadas_en_base(p_pol_x, x, base_x_L)   # [10/3, 1, 2/3]^T
print("\n=== (c) p^P(x) en base de Legendre {P0,P1,P2} ===")
print("Coeficientes (α0, α1, α2):", [sp.simplify(v) for v in a_L])
print("Verificación p^P - combinación (debe dar 0):",
      sp.simplify(p_pol_x - (a_L[0]*P0x + a_L[1]*P1x + a_L[2]*P2x)))

# ---------------- (d): Legendre en x e y ----------------
P0y, P1y, P2y = sp.legendre(0, y), sp.legendre(1, y), sp.legendre(2, y)
base_y_L = [P0y, P1y, P2y]

b_L = coordenadas_en_base(p_pol_y, y, base_y_L)   # [1, 1, 0]^T
C_L = producto_exterior(a_L, b_L)                 # Matriz c~^{ij}
p_tensor_L = tensor_desde_coeficientes(C_L, base_x_L, base_y_L)

print("\n=== (d) Base de Legendre en ambos espacios ===")
print("Coeficientes de p^G(y) en {P0,P1,P2}:", list(b_L))
print("Matriz c~^{ij} (i para {P0,P1,P2} en x; j para {P0,P1,P2} en y):")
sp.pprint(C_L)
print("\nVerificación p_tensor - p_tensor_L (debe ser 0):",
      sp.simplify(p_tensor - p_tensor_L))



=== (a) y (b) Base monomial {1,x,x^2} ⊗ {1,y,y^2} ===
Coeficientes de p^P(x): [3, 1, 1]
Coeficientes de p^G(y): [1, 1, 0]
Matriz c^{ij} (i para x^i; j para y^j):
⎡3  3  0⎤
⎢       ⎥
⎢1  1  0⎥
⎢       ⎥
⎣1  1  0⎦

Polinomio tensor p^{P⊗G}(x,y) expandido:
 2      2                    
x ⋅y + x  + x⋅y + x + 3⋅y + 3

=== (c) p^P(x) en base de Legendre {P0,P1,P2} ===
Coeficientes (α0, α1, α2): [10/3, 1, 2/3]
Verificación p^P - combinación (debe dar 0): 0

=== (d) Base de Legendre en ambos espacios ===
Coeficientes de p^G(y) en {P0,P1,P2}: [1, 1, 0]
Matriz c~^{ij} (i para {P0,P1,P2} en x; j para {P0,P1,P2} en y):
⎡10/3  10/3  0⎤
⎢             ⎥
⎢ 1     1    0⎥
⎢             ⎥
⎣2/3   2/3   0⎦

Verificación p_tensor - p_tensor_L (debe ser 0): 0
